<center><h1>Blombach_George_HW4</h1></center>
<br>
<br>

Name: George Blombach
Github Username: gblombach
USC ID: 2416096199

## 1. Time Series Classification Part 1: Feature Creation/Extraction (HW3 Rerun)

### (a) Obtain Data

Import packages

In [1]:
# pip install numpy pandas
import os
import numpy as np
import pandas as pd

from scipy.stats import bootstrap

Get the AReM Data Set

In [2]:
# initialize
path = "../data"
test_data = pd.DataFrame()
train_data = pd.DataFrame()
feature = 0 #for adding feature number to dataset

### (b) Splitting Data

In [3]:
# iterate and load datasets
for folder in os.scandir(path):
    if folder.is_dir():
        for file in os.scandir(folder):
            if folder.name == 'bending1' or folder.name == 'bending2':
                #extract data
                csvdata = pd.read_csv(os.path.join(path,folder.name,file.name),skiprows=4, usecols=range(7))
                if file.name == 'dataset1.csv' or file.name == 'dataset2.csv':
                    #add instance number
                    feature += 1
                    csvdata['Instance'] = feature
                    #add to testing dataframe
                    test_data = pd.concat([test_data, csvdata], ignore_index = True)
                else:
                    #add instance number
                    feature += 1
                    csvdata['Instance'] = feature
                    #add to training dataframe
                    #print(folder.name, file.name, len(csvdata))
                    train_data = pd.concat([train_data, csvdata], ignore_index = True)
            elif file.name == 'dataset1.csv' or file.name == 'dataset2.csv' or file.name == 'dataset3.csv':
                #print("test", folder.name, file.name)
                csvdata = pd.read_csv(os.path.join(path,folder.name,file.name),skiprows=4, usecols=range(7))
                #print(csvdata)
                #add instance number
                feature += 1
                csvdata['Instance'] = feature
                #add to dataframe
                test_data = pd.concat([test_data, csvdata], ignore_index = True)
            else:
                csvdata = pd.read_csv(os.path.join(path,folder.name,file.name),skiprows=4, usecols=range(7))
                #print(folder.name, file.name, len(csvdata))
                #print(csvdata)
                #add instance number
                feature += 1
                csvdata['Instance'] = feature
                #add to dataframe
                train_data = pd.concat([train_data, csvdata], ignore_index = True)
                #print("train", folder.name, file.name)
            #print(folder.name, file.name, feature)
#ensure numeric
train_data = train_data.apply(pd.to_numeric, errors='coerce')
test_data = test_data.apply(pd.to_numeric, errors='coerce')

#print(test_data)
print("Train Data: ", train_data.shape)
print("Test Data: ", test_data.shape)

#use this to see all output rows
full_data = pd.concat([train_data, test_data], ignore_index=True)
print("Full Data: ", full_data.shape)

#full_data.head()
pd.set_option('display.max_rows', None)
#print(test_data['Instance'].value_counts().sort_index())

Train Data:  (33119, 8)
Test Data:  (9120, 8)
Full Data:  (42239, 8)


### (c) Feature Extraction

In [4]:
chart = pd.DataFrame()
grouped_by_instance = full_data.groupby('Instance')
#print(grouped_by_instance)
#grouped_by_instance = full_data.groupby('Instance').mean()
#print(grouped_by_instance.head())

features = ['min', 'max', 'mean', 'median', 'std', '1stQrt', '3rdQrt']

#for i in range(len(grouped_by_instance)):
#    print(chart[grouped_by_instance].mean(axis=1))
#    for j in range(6):
#        chart[f'mean{j}'] = grouped_by_instance.iloc[j+1]
    #print(grouped_by_instance.iloc[1])

for j in range (88):
    chart.loc[j, "Instance"] = j+1
    for i in range (1,7):
        chart.loc[j, f'{features[0]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].min().values
        chart.loc[j, f'{features[1]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].max().values
        chart.loc[j, f'{features[2]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].mean().values
        chart.loc[j, f'{features[3]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].median().values
        chart.loc[j, f'{features[4]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].std().values
        chart.loc[j, f'{features[5]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].quantile(.25).values
        chart.loc[j, f'{features[6]}{i}'] = grouped_by_instance.get_group(j+1).iloc[:,i:i+1].quantile(.75).values
#fix integar in Instance col
chart["Instance"] = chart["Instance"].astype(int)

#display chart without index, rounded to 3 places
display(chart.style
        .format(precision=3) # round it
        .hide(axis='index') # hide ordinal
        .set_properties(**{'text-align': 'center'}) # center the data in the columns
       )


Instance,min1,max1,mean1,median1,std1,1stQrt1,3rdQrt1,min2,max2,mean2,median2,std2,1stQrt2,3rdQrt2,min3,max3,mean3,median3,std3,1stQrt3,3rdQrt3,min4,max4,mean4,median4,std4,1stQrt4,3rdQrt4,min5,max5,mean5,median5,std5,1stQrt5,3rdQrt5,min6,max6,mean6,median6,std6,1stQrt6,3rdQrt6
1,37.250,45.000,40.625,40.500,1.477,39.250,42.000,0.000,1.300,0.359,0.430,0.323,0.000,0.500,4.000,29.500,19.041,19.250,4.463,16.000,23.250,0.000,7.230,0.833,0.500,0.966,0.000,1.120,27.250,38.250,34.311,35.000,2.188,33.000,36.000,0.000,1.920,0.571,0.430,0.583,0.000,1.300
2,38.000,45.670,42.813,42.500,1.436,42.000,43.670,0.000,1.220,0.372,0.470,0.289,0.000,0.500,2.000,29.500,20.097,21.000,3.894,19.000,22.250,0.000,5.760,0.876,0.500,1.047,0.000,1.145,27.670,38.500,33.025,33.000,1.995,32.000,34.500,0.000,3.110,0.571,0.430,0.601,0.000,1.300
3,35.000,47.400,43.955,44.330,1.559,43.000,45.000,0.000,1.700,0.426,0.470,0.339,0.000,0.500,6.500,29.750,22.122,23.000,3.031,19.750,24.000,0.000,4.440,0.497,0.430,0.551,0.000,0.830,29.000,38.500,35.588,36.000,2.000,35.362,36.500,0.000,1.790,0.493,0.430,0.514,0.000,0.940
4,33.000,47.750,42.180,43.500,3.671,39.150,45.000,0.000,3.000,0.696,0.500,0.631,0.000,1.120,8.500,30.000,22.184,23.000,3.810,20.500,24.372,0.000,5.150,0.990,0.830,0.954,0.430,1.300,20.000,38.670,33.494,35.000,3.849,30.457,36.330,0.000,2.180,0.614,0.500,0.524,0.000,1.000
5,33.000,45.750,41.678,41.750,2.243,41.330,42.750,0.000,2.830,0.536,0.500,0.405,0.430,0.710,3.000,28.250,19.007,19.125,4.087,16.500,22.062,0.000,6.420,0.842,0.500,0.929,0.430,1.120,23.670,37.500,29.857,30.000,2.411,28.457,31.250,0.000,1.790,0.383,0.430,0.389,0.000,0.500
6,37.000,48.000,43.455,43.250,1.386,42.500,45.000,0.000,1.580,0.378,0.470,0.316,0.000,0.500,5.750,27.000,15.793,15.000,3.848,13.000,18.270,0.000,10.030,0.849,0.500,0.996,0.430,1.120,8.000,33.500,23.035,23.500,2.489,22.250,24.000,0.000,5.260,0.680,0.500,0.623,0.430,0.870
7,36.250,48.000,43.969,44.500,1.618,43.310,44.670,0.000,1.500,0.413,0.470,0.263,0.430,0.500,1.500,26.330,15.868,16.250,3.742,14.250,18.000,0.000,5.170,0.666,0.470,0.789,0.000,0.940,11.330,30.750,22.104,21.670,3.318,20.500,23.750,0.000,2.960,0.555,0.490,0.488,0.000,0.830
8,12.750,51.000,24.563,24.250,3.738,23.188,26.500,0.000,6.870,0.591,0.430,0.837,0.000,0.710,0.000,25.330,19.121,20.250,4.300,18.000,22.000,0.000,6.760,0.737,0.470,0.927,0.000,0.870,10.670,30.000,23.493,23.750,3.694,20.500,27.000,0.000,4.970,0.700,0.500,0.694,0.430,0.870
9,0.000,42.750,27.465,28.000,3.584,25.500,30.000,0.000,7.760,0.450,0.430,0.767,0.000,0.500,7.500,35.000,20.843,20.750,3.826,18.000,22.500,0.000,5.760,0.780,0.500,0.904,0.000,0.940,0.000,33.000,17.618,18.000,5.054,15.000,20.750,0.000,6.760,1.122,0.830,1.012,0.470,1.300
10,21.000,50.000,32.586,33.000,6.238,26.188,34.500,0.000,9.900,0.516,0.430,0.790,0.000,0.823,0.000,28.250,13.942,14.250,5.883,10.000,18.000,0.000,7.400,0.869,0.500,1.027,0.430,1.120,5.000,33.750,20.346,19.585,5.032,17.670,23.500,0.000,13.610,1.162,0.830,1.333,0.470,1.300


## 2. Time Series Classification Part 2: Binary and Multiclass Classification

### (a) Binary Classification Using Logistic Regression

#### i. Plots

Assume that you want to use the training set to classify bending from other
activities, i.e. you have a binary classification problem. Depict scatter plots
of the features you specified in 1(c)iv extracted from time series 1, 2, and 6 of
each instance, and use color to distinguish bending vs. other activities. (See
p. 129 of the textbook).

#### ii. Splitted Plots

#### iii. Time Series Segments

#### iv. Confusion Matrix

#### v. Test Classifier

#### vi. Separation

#### vii. Imbalance

### (b) Binary Classification Using L1-penalized logistic regression

#### i. Time Series Segments

#### ii. Comparison

### (c) Multi-class Classification (The Realistic Case)

#### i. Time Series Segments

#### ii. Naive Bayes

#### ii. Comparison

## 3. ISLR 4.8.3

This problem relates to the QDA model, in which the observations within each class are drawn from a normal distribution with a class-specific
mean vector and a class specific covariance matrix. We consider the simple case where p = 1; i.e. there is only one feature.

Suppose that we have K classes, and that if an observation belongs to the kth class then X comes from a one-dimensional normal distribution,
X ∼ N(μk, σ2k). Recall that the density function for the one-dimensional normal distribution is given in (4.16). Prove that in this case, the Bayes classifier is not linear. Argue that it is in fact quadratic.

*Hint: For this problem, you should follow the arguments laid out in Section 4.4.1, but without making the assumption that σ21 = . . . = σ2K*

## 4. ISLR 4.8.7

Suppose that we wish to predict whether a given stock will issue a dividend this year (“Yes” or “No”) based on X, last year’s percent profit. We examine a large number of companies and discover that the mean value of X for companies that issued a dividend was ¯X = 10, while the mean for those that didn’t was ¯X = 0. In addition, the variance of X for these two sets of companies was ˆσ2 = 36. Finally, 80 % of companies issued dividends. Assuming that X follows a normal distribution, predict the probability that a company will issue a dividend this year given that its percentage profit was X = 4 last year.

*Hint: Recall that the density function for a normal random variable is f(x) = 1 √2πσ2 e−(x−μ)2/2σ2 . You will need to use Bayes’ theorem.*

## 5.1. Extra Practice ISLR 4.8.4

## 5.2. Extra Practice ISLR 4.8.9